# 📖 Novel-TUI — Servidor LLM Remoto (GPU T4 en Google Colab)

Este cuaderno ejecuta **KoboldCpp** con aceleración CUDA en GPU T4 y el modelo **L3-8B-Stheno-v3.2** (GGUF Q5_K_M sin censura).

### ⚡ Ventajas:
1. **Persistencia en Google Drive:** El modelo se descarga una sola vez en tu Drive (`/NovelTUI_Models/`) y en los próximos arranques inicia en 5 segundos.
2. **Túnel Cloudflare Gratuito:** Genera una URL pública `https://*.trycloudflare.com/v1` compatible con la API de OpenAI para Novel-TUI.

In [ ]:
#@title 🚀 Iniciar Servidor KoboldCpp con GPU y Google Drive
import os
from google.colab import drive

# 1. Montar Google Drive para no descargar el modelo cada vez
drive.mount('/content/drive')

# 2. Configurar rutas
DRIVE_DIR = '/content/drive/MyDrive/NovelTUI_Models'
MODEL_FILE = f'{DRIVE_DIR}/L3-8B-Stheno-v3.2-Q5_K_M.gguf'
MODEL_URL = 'https://huggingface.co/bartowski/L3-8B-Stheno-v3.2-GGUF/resolve/main/L3-8B-Stheno-v3.2-Q5_K_M.gguf'
KOBOLD_URL = 'https://github.com/LostRuins/koboldcpp/releases/download/v1.78/koboldcpp-linux-x64-cuda1200'

!mkdir -p /content/novel-llm
!mkdir -p {DRIVE_DIR}
%cd /content/novel-llm

# 3. Descargar KoboldCpp si no existe
if not os.path.exists('/content/novel-llm/koboldcpp'):
    print('📥 Descargando KoboldCpp CUDA...')
    !wget -q -c {KOBOLD_URL} -O koboldcpp
    !chmod +x koboldcpp

# 4. Verificar o descargar el modelo en Google Drive
if os.path.exists(MODEL_FILE):
    print('⚡ Modelo encontrado en Google Drive. Creando enlace...')
    !ln -sf '{MODEL_FILE}' /content/novel-llm/model.gguf
else:
    print('📥 Descargando L3-8B-Stheno a Google Drive (solo la primera vez)...')
    !wget -c '{MODEL_URL}' -O '{MODEL_FILE}'
    !ln -sf '{MODEL_FILE}' /content/novel-llm/model.gguf

# 5. Iniciar KoboldCpp con todas las capas en GPU y Cloudflare tunnel
print('🚀 Iniciando servidor con GPU y Túnel Cloudflare...')
!./koboldcpp --model model.gguf --usecublas --gpulayers 33 --contextsize 8192 --usecloudflare --skiplauncher